# Card Embedding Sanity Check

Loads the raw OpenAI/Azure text embeddings, reduces dimensionality with PCA
(and shows how much variance is retained at various component counts so we
can pick an informed cutoff instead of guessing), projects the top principal
components down to 2D with UMAP, and renders an interactive scatterplot
(hover over a point to preview the card's name + oracle text) for spot-checking
whether functionally/semantically similar cards actually cluster together.

In [1]:
from pathlib import Path

from magicnet.embedding_viz import (
    load_embeddings,
    load_card_metadata,
    fit_pca,
    cumulative_explained_variance,
    n_components_for_variance,
    fit_umap_2d,
    build_plot_frame,
)

DATA_DIR = Path("../data/processed")
embeddings, ids = load_embeddings(DATA_DIR / "card_embeddings.npy", DATA_DIR / "card_embeddings.ids.json")
card_meta = load_card_metadata(DATA_DIR / "cards.jsonl")
print(f"{embeddings.shape[0]} cards, {embeddings.shape[1]}-dim raw embeddings")

33810 cards, 3072-dim raw embeddings


## How many principal components do we actually need?

Fit PCA out to a generous number of components once, then look at the
cumulative explained-variance curve. This tells us concretely how much
information we'd be throwing away at any given cutoff, rather than assuming
"half the original dimension" is the right number.

In [2]:
import plotly.express as px

MAX_COMPONENTS = 1536  # half of the raw 3072-dim embedding; upper bound for this scan
_, pca_full = fit_pca(embeddings, n_components=MAX_COMPONENTS)
cum_variance = cumulative_explained_variance(pca_full)

fig = px.line(
    x=range(1, len(cum_variance) + 1),
    y=cum_variance,
    labels={"x": "Number of principal components", "y": "Cumulative explained variance"},
    title="PCA cumulative explained variance",
)
fig.add_hline(y=0.90, line_dash="dot", annotation_text="90%")
fig.show()

for target in (0.80, 0.85, 0.90, 0.95):
    print(f"{target*100:.0f}% variance -> {n_components_for_variance(pca_full, target)} components")

80% variance -> 300 components
85% variance -> 419 components
90% variance -> 616 components
95% variance -> 1005 components


On this dataset, ~85-90% of the variance is captured well under half of the
raw dimensionality (see printed counts above) -- the curve flattens out hard
past a few hundred components. **`N_COMPONENTS` below defaults to 512** (a
round number that lands around ~88% cumulative variance here); adjust it
based on what the printed table above actually shows if you want to trade off
more/less compression.

In [3]:
N_COMPONENTS = 512

reduced_embeddings, pca = fit_pca(embeddings, n_components=N_COMPONENTS)
print(f"Reduced to {reduced_embeddings.shape[1]} dims, "
      f"retaining {cumulative_explained_variance(pca)[-1]*100:.1f}% of variance")

Reduced to 512 dims, retaining 87.6% of variance


## UMAP projection to 2D, for visualization only

In [4]:
coords_2d = fit_umap_2d(reduced_embeddings)
plot_df = build_plot_frame(coords_2d, ids, card_meta)
plot_df.head()

/home/john/Projects/magicnet/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,oracle_id,x,y,name,mana_cost,cmc,type_line,colors,color_identity,layout,text,hover_text,color_group
0,b34bb2dc-c1af-4d77-b0b3-a0fb342a5fc6,4.061090,7.479915,Forest,,0.0,Basic Land — Forest,[],[G],normal,Forest | Basic Land — Forest | ({T}: Add {G}.),Forest | Basic Land — Forest | ({T}: Add {G}.),Colorless
1,44623693-51d6-49ad-8cd7-140505caf02f,-4.060372,-2.408318,Fury Sliver,{5}{R},6.0,Creature — Sliver,[R],[R],normal,Fury Sliver | {5}{R} | Creature — Sliver | All...,Fury Sliver | {5}{R} | Creature — Sliver | All...,Red
2,8ae3562f-28b7-4462-96ed-be0cf7052ccc,2.279816,0.210295,Kor Outfitter,{W}{W},2.0,Creature — Kor Soldier,[W],[W],normal,Kor Outfitter | {W}{W} | Creature — Kor Soldie...,Kor Outfitter | {W}{W} | Creature — Kor Soldie...,White
3,9f0d82ae-38bf-45d8-8cda-982b6ead1d72,-0.403999,1.220404,Siren Lookout,{2}{U},3.0,Creature — Siren Pirate,[U],[U],normal,Siren Lookout | {2}{U} | Creature — Siren Pira...,Siren Lookout | {2}{U} | Creature — Siren Pira...,Blue
4,5aa12aff-db3c-4be5-822b-3afdf536b33e,-2.386089,6.045058,Web,{G},1.0,Enchantment — Aura,[G],[G],normal,Web | {G} | Enchantment — Aura | Enchant creat...,Web | {G} | Enchantment — Aura | Enchant creat...,Green


## Interactive scatterplot

Hover over any point to preview the card's name and (truncated) oracle text.
Colored by color identity as a quick, independent visual sanity check --
if the embedding space is meaningful, you'd expect at least some loose
color-based clustering (color identity is heavily correlated with mechanical
themes), on top of whatever finer functional clusters exist within a color.

In [5]:
fig = px.scatter(
    plot_df,
    x="x",
    y="y",
    color="color_group",
    hover_name="name",
    hover_data={"hover_text": True, "type_line": True, "x": False, "y": False, "color_group": False},
    opacity=0.6,
    title=f"UMAP projection of top-{N_COMPONENTS} PCA components ({len(plot_df)} cards)",
    height=800,
)
fig.update_traces(marker=dict(size=4))
fig.show()

## Spot-check: nearest neighbors of a specific card

A scatterplot is a lossy 2D summary; for an actual nearest-neighbor sanity
check (e.g. "does Fatal Push cluster near other cheap removal, not near
unrelated cards"), query the *full-precision* reduced embeddings directly.

In [6]:
import numpy as np

def nearest_neighbors(card_name: str, k: int = 10):
    matches = card_meta[card_meta["name"] == card_name]
    if matches.empty:
        raise ValueError(f"No card named {card_name!r} found")
    idx = ids.index(matches.index[0])
    query = reduced_embeddings[idx]
    sims = reduced_embeddings @ query / (
        np.linalg.norm(reduced_embeddings, axis=1) * np.linalg.norm(query) + 1e-9
    )
    top = np.argsort(-sims)[: k + 1]
    return [(ids[i], card_meta.loc[ids[i], "name"], sims[i]) for i in top if ids[i] != matches.index[0]][:k]

for oracle_id, name, sim in nearest_neighbors("Fatal Push"):
    print(f"{sim:.3f}  {name}")

0.479  Eliminate
0.467  Vanquish the Weak
0.451  Easy Prey
0.436  Defeat
0.421  Smother
0.417  Cast Down
0.411  Murder
0.408  Abrupt Decay
0.405  Depressurize
0.404  Mob
